In [23]:
import numpy as np
import torch
import random
import os
import json
import argparse

from torch.nn import Parameter
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid
from pathlib import Path
from scipy.interpolate import interp1d
from dolfin import *

from torch.optim import Adam
from gpytorch.optim import NGD

from alfi.models import TrainMode, MultiOutputGP, PartialLFM, generate_multioutput_gp
from alfi.impl.pdes import ReactionDiffusion
from alfi.utilities.data import dros_ground_truth
from alfi.utilities.fenics import interval_mesh
from alfi.datasets import DrosophilaSpatialTranscriptomics, HomogeneousReactionDiffusion
from alfi.trainers import PartialPreEstimator, PDETrainer
from alfi.plot import plot_spatiotemporal_data, Plotter1d
from alfi.utilities.torch import softplus, inv_softplus, q2, cia, smse
from alfi.configuration import VariationalConfiguration


In [24]:
experiment_name = "gt_46_1753639811.38302"
SEED = 46

In [25]:
plt.rcParams['font.family'] = 'DeJavu Serif'
plt.rcParams['font.serif'] = ['Times New Roman']

def evaluate_model(lfm, trainer):
    tx = trainer.tx
    disc = trainer.disc
    h = lfm(tx, step=disc)
    f_mean = h.mean.detach()
    f_var = h.variance.detach()
    y_target = trainer.y_target[0]
    f_mean_test = f_mean.squeeze()
    f_var_test = f_var.squeeze()

    prot_q2 = q2(y_target, f_mean.squeeze()).item()
    prot_cia = cia(y_target, f_mean_test, f_var_test).item()
    prot_smse = smse(y_target, f_mean_test).mean().item()

    f = lfm.gp_model(trainer.tx.t())
    lf_target = trainer.lf_target.squeeze()
    num_t = trainer.num_t
    num_x = trainer.num_x
    f_mean = f.mean.view(num_t, num_x)[::disc].reshape(-1)
    f_var = f.variance.view(num_t, num_x)[::disc].reshape(-1)

    mrna_q2 = q2(lf_target, f_mean.squeeze()).item()
    mrna_cia = cia(lf_target, f_mean.squeeze(), f_var.squeeze()).item()
    mrna_smse = smse(lf_target, f_mean.squeeze()).mean().item()

    return prot_q2, prot_cia, prot_smse, mrna_q2, mrna_cia, mrna_smse

In [26]:
gene = "gt"
use_lhs = True
train_ratio=0.3
skip=True
learn_inducing_locations=False
gp_use_tril=True
params_grad=False
experiment_dir="experiment_results"

dataset = DrosophilaSpatialTranscriptomics(
    gene=gene, data_dir='./data', scale=True, disc=20) # data folder
params = dict(lengthscale=10,
        **dros_ground_truth(gene),
        parameter_grad=False,
        natural=False,
        clamp=True)
num_epochs = 80

disc = dataset.disc

data = next(iter(dataset))
tx, y_target = data
lengthscale = params['lengthscale']
zero_mean = params['zero_mean'] if 'zero_mean' in params else False

#### Generate FENiCS 1D mesh from the data
# Alternatively, if our spatial data is not uniformly spaced, we can define a custom mesh as follows.
# We calculate a mesh that contains all possible spatial locations in the dataset

spatial = np.unique(tx[1, :])
mesh = interval_mesh(spatial)


if tx.shape[1] > 1000:
    num_inducing = int(tx.shape[1] * 3/6)
else:
    num_inducing = int(tx.shape[1] * 5/6)
# whether inducing points should be sampled using LHS or random choice
if use_lhs:
    print('tx', tx.shape)
    from smt.sampling_methods import LHS
    ts = tx[0, :].unique().sort()[0].numpy()
    xs = tx[1, :].unique().sort()[0].numpy()
    xlimits = np.array([[ts[0], ts[-1]],[xs[0], xs[-1]]])
    sampling = LHS(xlimits=xlimits, random_state=np.random.RandomState(SEED))
    inducing_points = torch.tensor(sampling(num_inducing)).unsqueeze(0)
else:
    inducing_points = torch.stack([
        tx[0, torch.randperm(tx.shape[1], generator=torch.Generator().manual_seed(SEED))[:num_inducing]],
        tx[1, torch.randperm(tx.shape[1], generator=torch.Generator().manual_seed(SEED))[:num_inducing]]
    ], dim=1).unsqueeze(0)
gp_kwargs = dict(learn_inducing_locations=learn_inducing_locations,
                natural=params['natural'],
                use_tril=gp_use_tril)


ts = tx[0, :].unique().sort()[0].numpy()
t_range = (ts[0], ts[-1])
# :HP: number of time steps to discretise the PDE of the dataset
time_steps = dataset.num_discretised
fenics_model = ReactionDiffusion(t_range, time_steps, mesh)

# :HP: configuration for the variational inference, initial conditions, number of samples
config = VariationalConfiguration(
    initial_conditions=False,
    num_samples=5
)

# Set up the LFM
# include DE parameters
parameter_grad = params['parameter_grad'] if 'parameter_grad' in params else True
sensitivity = Parameter(
    inv_softplus(torch.tensor(params['sensitivity'])) * torch.ones((1, 1), dtype=torch.float64),
    requires_grad=False)
decay = Parameter(
    inv_softplus(torch.tensor(params['decay'])) * torch.ones((1, 1), dtype=torch.float64),
    requires_grad=parameter_grad)
diffusion = Parameter(
    inv_softplus(torch.tensor(params['diffusion'])) * torch.ones((1, 1), dtype=torch.float64),
    requires_grad=parameter_grad)

fenics_params = [sensitivity, decay, diffusion]
num_training = int(train_ratio * tx.shape[1])


tx torch.Size([2, 512])


In [27]:
gp_args = [1, inducing_points]
gp_kwargs = dict(learn_inducing_locations=learn_inducing_locations,
                natural=params['natural'],
                use_tril=gp_use_tril)
gp_model = generate_multioutput_gp(
    *gp_args,
    initial_lengthscale=lengthscale,
    ard_dims=2,
    zero_mean=zero_mean,
    gp_kwargs=gp_kwargs)
gp_args = [gp_model.mean_module, gp_model.covar_module] + [gp_args[1], gp_args[0]]
filepath = str(num_epochs) +"epochs_trained_"   # Specify the file path to save the model
orig_data = dataset.orig_data.squeeze().t()
lfm = PartialLFM.load(experiment_dir+"/"+ experiment_name + "/" + filepath,
                    gp_cls=MultiOutputGP,
                    gp_args=gp_args,
                    gp_kwargs=gp_kwargs,
                    lfm_args=[1, fenics_model, fenics_params, config])
# lfm = PartialLFM(1, gp_model, fenics_model, fenics_params, config)

gp_model = lfm.gp_model
optimizer = [torch.optim.Adam(lfm.parameters(), lr=0.07)]
trainer = PDETrainer(lfm, optimizer, dataset, track_parameters=list(lfm.fenics_named_parameters.keys()), lf_target=orig_data)
lfm.eval()
prot_q2, prot_cia, prot_smse, mrna_q2, mrna_cia, mrna_smse = evaluate_model(lfm, trainer)
result_dict = {
    'prot_q2': prot_q2,
    'prot_cia': prot_cia,
    'prot_smse': prot_smse,
    'mrna_q2': mrna_q2,
    'mrna_cia': mrna_cia,
    'mrna_smse': mrna_smse
}
with open(experiment_dir + "/" + experiment_name + "/result.json", "w") as f:
    json.dump(result_dict, f, indent=4)

x dp is set to tensor(1., dtype=torch.float64)


### Make table

In [29]:
import os
import json
import numpy as np
import csv

# Directory containing experiment results
experiment_dir = "experiment_results"

# Initialize dictionaries to store results
results = {"gt": {}, "kni": {}}

# Iterate through subdirectories
for subdir in os.listdir(experiment_dir):
    if subdir.startswith("gt_") or subdir.startswith("kni_"):
        prefix = "gt" if subdir.startswith("gt_") else "kni"
        result_file = os.path.join(experiment_dir, subdir, "result.json")
        
        if os.path.exists(result_file):
            with open(result_file, "r") as f:
                data = json.load(f)
            
            for key, value in data.items():
                if key not in results[prefix]:
                    results[prefix][key] = []
                results[prefix][key].append(value)

# Compute mean and standard deviation
stats = {"gt": {}, "kni": {}}
for prefix in results:
    for key, values in results[prefix].items():
        stats[prefix][key] = {
            "mean": np.mean(values),
            "std": np.std(values)
        }

# Print results
# Specify the output CSV file
output_csv = "experiment_stats.csv"

# Write stats to a CSV file
with open(output_csv, mode="w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    # Write the header
    header = ["gene"] + list(next(iter(stats.values())).keys())
    writer.writerow(header)
    # Write the data
    for prefix, metrics in stats.items():
        row = [prefix] + [f"{values['mean']:.4f} ± {values['std']:.4f}" for values in metrics.values()]
        writer.writerow(row)

print(f"Stats have been written to {output_csv}")

Stats have been written to experiment_stats.csv


In [40]:
import pandas as pd
df = pd.read_csv("paper_results.csv")
# Split columns with " ± " into separate mean and standard deviation columns
for col in df.columns:
    if df[col].dtype == object and df[col].str.contains(" ± ").any():
        df[[f"{col}_mean", f"{col}_sd"]] = df[col].str.split(" ± ", expand=True).astype(float)
        df.drop(columns=[col], inplace=True)

# Adjust "cia_mean" columns and divide all numeric entries by 100
for col in df.columns:
    if col.endswith("cia_mean"):
        df[col] = (68 - df[col]) / 100
    elif df[col].dtype in [float, int]:
        df[col] = df[col] / 100

In [43]:
df2 = pd.read_csv("experiment_stats.csv")
for col in df2.columns:
    if df2[col].dtype == object and df2[col].str.contains(" ± ").any():
        df2[[f"{col}_mean", f"{col}_sd"]] = df2[col].str.split(" ± ", expand=True).astype(float)
        df2.drop(columns=[col], inplace=True)

In [48]:
df

,model,gene,mrna_q2_mean,mrna_q2_sd,prot_q2_mean,prot_q2_sd,mrna_cia_mean,mrna_cia_sd,prot_cia_mean,prot_cia_sd
0,gp-mrna,kr,0.905,0.020,0.908,0.006,0.717,0.067,0.462,0.040
1,gp-mrna,kni,0.811,0.025,0.887,0.008,0.497,0.063,0.327,0.053
2,gp-mrna,gt,0.912,0.019,0.923,0.006,0.687,0.033,0.422,0.018
3,alfi,kr,0.824,0.012,0.957,0.002,0.647,0.035,0.486,0.020
4,alfi,kni,0.772,0.026,0.869,0.029,0.695,0.016,0.556,0.091
5,alfi,gt,0.724,0.033,0.865,0.012,0.719,0.044,0.622,0.143


In [51]:
df.to_csv("paper_results.csv", index=False)
df2.to_csv("experiment_stats.csv", index=False)

In [54]:
df.columns = df.columns.str.strip()
df2.columns = df2.columns.str.strip()

In [55]:
# Find shared columns between df and df2
shared_columns = df.columns.intersection(df2.columns)

# Concatenate the dataframes by stacking rows for shared columns
stacked_df = pd.concat([df[shared_columns], df2[shared_columns]], ignore_index=True)

print(stacked_df)

     model  gene  mrna_q2_mean  mrna_q2_sd  prot_q2_mean  prot_q2_sd  \
0  gp-mrna    kr        0.9050      0.0200        0.9080      0.0060   
1  gp-mrna   kni        0.8110      0.0250        0.8870      0.0080   
2  gp-mrna    gt        0.9120      0.0190        0.9230      0.0060   
3     alfi    kr        0.8240      0.0120        0.9570      0.0020   
4     alfi   kni        0.7720      0.0260        0.8690      0.0290   
5     alfi    gt        0.7240      0.0330        0.8650      0.0120   
6     repr    gt        0.7106      0.0301        0.6849      0.0350   
7     repr   kni        0.4874      0.0549        0.8679      0.0153   

   mrna_cia_mean  mrna_cia_sd  prot_cia_mean  prot_cia_sd  
0         0.7170       0.0670          0.462       0.0400  
1         0.4970       0.0630          0.327       0.0530  
2         0.6870       0.0330          0.422       0.0180  
3         0.6470       0.0350          0.486       0.0200  
4         0.6950       0.0160          0.556       

In [56]:
stacked_df.to_csv("experiment_stats.csv", index=False)

In [57]:
# Round numeric entries to 3 significant figures and combine mean and sd columns
for col in stacked_df.columns:
    if col.endswith("_mean"):
        base_col = col[:-5]
        sd_col = f"{base_col}_sd"
        if sd_col in stacked_df.columns:
            stacked_df[base_col] = stacked_df[col].round(3).astype(str) + " ± " + stacked_df[sd_col].round(3).astype(str)
            stacked_df.drop(columns=[col, sd_col], inplace=True)

# Save the processed table to a CSV file
stacked_df.to_csv("pre_process_table.csv", index=False)

https://tableconvert.com/csv-to-markdown#dataSource